In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm, t, chi2
from arch import arch_model

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

PORTFOLIO_VALUE = 1_000_000
CONFIDENCE_LEVEL = 0.99
BASELINE_WINDOW = 250
REFIT_EVERY = 5

COVID_START = pd.Timestamp("2020-02-01")
COVID_END = pd.Timestamp("2020-04-30")

CHECK_DATES = pd.to_datetime([
    "2011-01-03",
    "2018-02-06",
    "2020-03-16",
    "2020-03-25",
    "2025-04-04",
])


# ============================================================
# FIND REPOSITORY ROOT
# ============================================================

def find_repo_root():
    current = Path.cwd().resolve()

    for candidate in [current] + list(current.parents):
        if (candidate / "data" / "spy_daily_raw.csv").exists():
            return candidate

    raise FileNotFoundError("Could not locate repository root.")


ROOT = find_repo_root()

RAW_PATH = ROOT / "data" / "spy_daily_raw.csv"
GARCH_PATH = ROOT / "results" / "garch_var_output.csv"
COMBINED_PATH = ROOT / "results" / "combined_var_results.csv"

OUT_DIR = ROOT / "results" / "validation" / "garch"
FIG_DIR = OUT_DIR / "figures"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("GARCH validation output:", OUT_DIR)


# ============================================================
# LOAD AND PREPARE DATA
# ============================================================

raw = pd.read_csv(
    RAW_PATH,
    encoding="utf-8-sig"
)

raw.columns = [
    c.strip()
    for c in raw.columns
]

raw["Date"] = pd.to_datetime(
    raw["Date"]
)

raw = (
    raw
    .sort_values("Date")
    .reset_index(drop=True)
)

raw = (
    raw
    .dropna(subset=["Last Price"])
    .reset_index(drop=True)
)

raw = (
    raw.loc[
        raw.groupby("Date")["Last Price"].idxmax()
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

spy = raw[
    raw["Date"] >= pd.Timestamp("2010-01-01")
].copy()

spy["SPY_Return"] = (
    spy["Last Price"].pct_change()
)

spy["Portfolio_PnL"] = (
    PORTFOLIO_VALUE
    * spy["SPY_Return"]
)

spy = (
    spy
    .dropna(
        subset=[
            "SPY_Return",
            "Portfolio_PnL",
        ]
    )
    .reset_index(drop=True)
)

ret_df = spy.set_index("Date")


garch_original = pd.read_csv(
    GARCH_PATH
)

garch_original["Date"] = pd.to_datetime(
    garch_original["Date"]
)

garch_original = (
    garch_original
    .sort_values("Date")
    .reset_index(drop=True)
)


combined = pd.read_csv(
    COMBINED_PATH
)

combined["Date"] = pd.to_datetime(
    combined["Date"]
)

combined = (
    combined
    .sort_values("Date")
    .reset_index(drop=True)
)


print()
print("DATA SUMMARY")
print("--------------------------------")
print(
    f"SPY return observations: "
    f"{len(ret_df):,}"
)
print(
    f"GARCH forecasts: "
    f"{len(garch_original):,}"
)
print(
    f"First GARCH forecast: "
    f"{garch_original['Date'].min().date()}"
)
print(
    f"Last GARCH forecast: "
    f"{garch_original['Date'].max().date()}"
)


# ============================================================
# GARCH MODEL FUNCTION
# ============================================================

def fit_garch_series(
    window,
    distribution="StudentsT",
    refit_every=5,
    label="GARCH",
):

    rows = []
    params = None

    for i in range(
        window,
        len(ret_df)
    ):

        window_returns = (
            100
            * ret_df["SPY_Return"].iloc[
                i-window:i
            ]
        )

        model = arch_model(
            window_returns,
            mean="Constant",
            vol="Garch",
            p=1,
            q=1,
            dist=distribution,
        )

        should_refit = (
            params is None
            or
            (i - window) % refit_every == 0
        )

        if should_refit:

            try:
                fit = model.fit(
                    disp="off",
                    starting_values=(
                        None
                        if params is None
                        else params.values
                    ),
                    show_warning=False,
                )

            except Exception:

                fit = model.fit(
                    disp="off",
                    show_warning=False,
                )

            params = fit.params.copy()

        else:

            fit = model.fix(
                params
            )

        forecast = fit.forecast(
            horizon=1,
            reindex=False,
        )

        mu = (
            forecast
            .mean
            .values[-1, 0]
        )

        sigma = np.sqrt(
            forecast
            .variance
            .values[-1, 0]
        )

        if distribution.lower() == "normal":

            q = norm.ppf(
                1 - CONFIDENCE_LEVEL
            )

            nu = np.nan

        else:

            nu = fit.params["nu"]

            q = (
                t.ppf(
                    1 - CONFIDENCE_LEVEL,
                    nu,
                )
                * np.sqrt(
                    (nu - 2)
                    / nu
                )
            )

        var = (
            -(
                mu
                + sigma * q
            )
            / 100
            * PORTFOLIO_VALUE
        )

        rows.append({
            "Date": ret_df.index[i],
            "Portfolio_PnL": (
                ret_df["Portfolio_PnL"]
                .iloc[i]
            ),
            "SPY_Return": (
                ret_df["SPY_Return"]
                .iloc[i]
            ),
            "VaR": var,
            "Mean_pct": mu,
            "Sigma_pct": sigma,
            "Omega": fit.params.get(
                "omega",
                np.nan,
            ),
            "Alpha": fit.params[
                "alpha[1]"
            ],
            "Beta": fit.params[
                "beta[1]"
            ],
            "Nu": nu,
        })

        if (
            (i - window) % 750 == 0
        ):
            print(
                f"{label}: processed "
                f"{i-window:,} forecasts"
            )

    result = pd.DataFrame(
        rows
    )

    result["Persistence"] = (
        result["Alpha"]
        + result["Beta"]
    )

    return result


# ============================================================
# STATISTICAL FUNCTIONS
# ============================================================

def log_component(
    count,
    probability,
):
    if count == 0:
        return 0.0

    probability = np.clip(
        probability,
        1e-12,
        1 - 1e-12,
    )

    return (
        count
        * np.log(probability)
    )


def kupiec_test(
    exceptions,
    expected_rate=0.01,
):

    x = int(
        np.sum(exceptions)
    )

    n = len(exceptions)

    phat = x / n

    ll_null = (
        log_component(
            n - x,
            1 - expected_rate,
        )
        +
        log_component(
            x,
            expected_rate,
        )
    )

    ll_alt = (
        log_component(
            n - x,
            1 - phat,
        )
        +
        log_component(
            x,
            phat,
        )
    )

    lr = -2 * (
        ll_null
        - ll_alt
    )

    p_value = chi2.sf(
        lr,
        1,
    )

    return (
        lr,
        p_value,
    )


def christoffersen_test(
    exceptions
):

    x = np.asarray(
        exceptions,
        dtype=int,
    )

    previous = x[:-1]
    current = x[1:]

    n00 = int(
        np.sum(
            (previous == 0)
            &
            (current == 0)
        )
    )

    n01 = int(
        np.sum(
            (previous == 0)
            &
            (current == 1)
        )
    )

    n10 = int(
        np.sum(
            (previous == 1)
            &
            (current == 0)
        )
    )

    n11 = int(
        np.sum(
            (previous == 1)
            &
            (current == 1)
        )
    )

    pi01 = (
        n01
        / (n00 + n01)
        if (n00 + n01)
        else 0
    )

    pi11 = (
        n11
        / (n10 + n11)
        if (n10 + n11)
        else 0
    )

    total = (
        n00
        + n01
        + n10
        + n11
    )

    pi = (
        (n01 + n11)
        / total
        if total
        else 0
    )

    ll_independent = (
        log_component(
            n00 + n10,
            1 - pi,
        )
        +
        log_component(
            n01 + n11,
            pi,
        )
    )

    ll_markov = (
        log_component(
            n00,
            1 - pi01,
        )
        +
        log_component(
            n01,
            pi01,
        )
        +
        log_component(
            n10,
            1 - pi11,
        )
        +
        log_component(
            n11,
            pi11,
        )
    )

    lr = -2 * (
        ll_independent
        - ll_markov
    )

    p_value = chi2.sf(
        lr,
        1,
    )

    return {
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "LR": lr,
        "p_value": p_value,
    }


def model_stats(
    df,
    var_col,
):

    loss = (
        -df["Portfolio_PnL"]
    )

    exceptions = (
        loss
        > df[var_col]
    )

    kupiec_lr, kupiec_p = (
        kupiec_test(
            exceptions
        )
    )

    christ = (
        christoffersen_test(
            exceptions
        )
    )

    return {
        "Observations": len(df),
        "Average_VaR": (
            df[var_col].mean()
        ),
        "Maximum_VaR": (
            df[var_col].max()
        ),
        "Exceptions": int(
            exceptions.sum()
        ),
        "Exception_Rate": (
            exceptions.mean()
        ),
        "Kupiec_LR": kupiec_lr,
        "Kupiec_p": kupiec_p,
        "Christoffersen_LR": (
            christ["LR"]
        ),
        "Christoffersen_p": (
            christ["p_value"]
        ),
        "n00": christ["n00"],
        "n01": christ["n01"],
        "n10": christ["n10"],
        "n11": christ["n11"],
    }


# ============================================================
# IMPLEMENTATION CHECKS
# ============================================================

raw_project = raw[
    raw["Date"]
    >= pd.Timestamp("2010-01-01")
].copy()

implementation_checks = pd.DataFrame(
    [
        [
            "Raw dataset available",
            RAW_PATH.exists(),
        ],
        [
            "GARCH output available",
            GARCH_PATH.exists(),
        ],
        [
            "Combined benchmark output available",
            COMBINED_PATH.exists(),
        ],
        [
            "Project dates sorted",
            raw_project[
                "Date"
            ].is_monotonic_increasing,
        ],
        [
            "No duplicate project dates",
            not raw_project[
                "Date"
            ].duplicated().any(),
        ],
        [
            "No missing Last Price",
            raw_project[
                "Last Price"
            ].isna().sum() == 0,
        ],
        [
            "All project prices positive",
            (
                raw_project[
                    "Last Price"
                ] > 0
            ).all(),
        ],
        [
            "GARCH output dates sorted",
            garch_original[
                "Date"
            ].is_monotonic_increasing,
        ],
        [
            "No duplicate GARCH dates",
            not garch_original[
                "Date"
            ].duplicated().any(),
        ],
        [
            "Normal GARCH VaR positive",
            (
                garch_original[
                    "VaR_N"
                ] > 0
            ).all(),
        ],
        [
            "Student-t GARCH VaR positive",
            (
                garch_original[
                    "VaR_T"
                ] > 0
            ).all(),
        ],
        [
            "GARCH and benchmark row counts match",
            len(garch_original)
            == len(combined),
        ],
        [
            "GARCH and benchmark dates match",
            garch_original[
                "Date"
            ].equals(
                combined["Date"]
            ),
        ],
    ],
    columns=[
        "Check",
        "Result",
    ],
)

implementation_checks.to_csv(
    OUT_DIR
    / "garch_implementation_checks.csv",
    index=False,
)

print()
print("IMPLEMENTATION CHECKS")
print("--------------------------------")
print(
    implementation_checks
    .to_string(index=False)
)


# ============================================================
# REPRODUCE BASELINE STUDENT-T
# ============================================================

print()
print(
    "RECALCULATING BASELINE "
    "STUDENT-T GARCH..."
)

garch_t_250 = fit_garch_series(
    window=250,
    distribution="StudentsT",
    refit_every=5,
    label="Student-t 250-day",
)


# ============================================================
# REPRODUCE BASELINE NORMAL
# ============================================================

print()
print(
    "RECALCULATING BASELINE "
    "NORMAL GARCH..."
)

garch_n_250 = fit_garch_series(
    window=250,
    distribution="normal",
    refit_every=5,
    label="Normal 250-day",
)


# ============================================================
# BASELINE REPRODUCTION CHECK
# ============================================================

reproduction = (
    garch_original[
        [
            "Date",
            "VaR_N",
            "VaR_T",
        ]
    ]
    .merge(
        garch_n_250[
            [
                "Date",
                "VaR",
            ]
        ].rename(
            columns={
                "VaR":
                "Calculated_VaR_N"
            }
        ),
        on="Date",
        how="inner",
    )
    .merge(
        garch_t_250[
            [
                "Date",
                "VaR",
            ]
        ].rename(
            columns={
                "VaR":
                "Calculated_VaR_T"
            }
        ),
        on="Date",
        how="inner",
    )
)

reproduction[
    "Normal_Difference"
] = (
    reproduction[
        "Calculated_VaR_N"
    ]
    -
    reproduction[
        "VaR_N"
    ]
)

reproduction[
    "StudentT_Difference"
] = (
    reproduction[
        "Calculated_VaR_T"
    ]
    -
    reproduction[
        "VaR_T"
    ]
)

print()
print(
    "BASELINE REPRODUCTION CHECK"
)
print("--------------------------------")

print(
    "Maximum Normal difference:",
    f"${reproduction['Normal_Difference'].abs().max():,.10f}"
)

print(
    "Maximum Student-t difference:",
    f"${reproduction['StudentT_Difference'].abs().max():,.10f}"
)


# ============================================================
# INDEPENDENT DATE CHECKS
# ============================================================

independent_checks = (
    reproduction[
        reproduction[
            "Date"
        ].isin(
            CHECK_DATES
        )
    ]
    .copy()
)

independent_checks = (
    independent_checks[
        [
            "Date",
            "VaR_N",
            "Calculated_VaR_N",
            "Normal_Difference",
            "VaR_T",
            "Calculated_VaR_T",
            "StudentT_Difference",
        ]
    ]
)

independent_checks.to_csv(
    OUT_DIR
    / "garch_independent_date_checks.csv",
    index=False,
)

print()
print("INDEPENDENT DATE CHECKS")
print("--------------------------------")

print(
    independent_checks.to_string(
        index=False,
        formatters={
            "VaR_N":
                "${:,.2f}".format,
            "Calculated_VaR_N":
                "${:,.2f}".format,
            "Normal_Difference":
                "${:,.6f}".format,
            "VaR_T":
                "${:,.2f}".format,
            "Calculated_VaR_T":
                "${:,.2f}".format,
            "StudentT_Difference":
                "${:,.6f}".format,
        },
    )
)


# ============================================================
# BASELINE BACKTESTING
# ============================================================

baseline = (
    garch_original.copy()
)

baseline["Loss"] = (
    -baseline[
        "Portfolio_PnL"
    ]
)

baseline["Exception_N"] = (
    baseline["Loss"]
    >
    baseline["VaR_N"]
)

baseline["Exception_T"] = (
    baseline["Loss"]
    >
    baseline["VaR_T"]
)

summary_rows = []

for (
    name,
    var_col,
    exception_col,
) in [
    (
        "Normal GARCH(1,1)",
        "VaR_N",
        "Exception_N",
    ),
    (
        "Student-t GARCH(1,1)",
        "VaR_T",
        "Exception_T",
    ),
]:

    result = model_stats(
        baseline,
        var_col,
    )

    summary_rows.append({
        "Model": name,
        "Observations":
            result["Observations"],
        "Expected_Exceptions":
            result["Observations"]
            * 0.01,
        "Actual_Exceptions":
            result["Exceptions"],
        "Exception_Rate":
            result["Exception_Rate"],
        "Average_VaR":
            result["Average_VaR"],
        "Maximum_VaR":
            result["Maximum_VaR"],
    })

exception_summary = pd.DataFrame(
    summary_rows
)

exception_summary.to_csv(
    OUT_DIR
    / "garch_exception_summary.csv",
    index=False,
)

print()
print("BACKTESTING SUMMARY")
print("--------------------------------")

print(
    exception_summary.to_string(
        index=False,
        formatters={
            "Expected_Exceptions":
                "{:.2f}".format,
            "Exception_Rate":
                "{:.2%}".format,
            "Average_VaR":
                "${:,.2f}".format,
            "Maximum_VaR":
                "${:,.2f}".format,
        },
    )
)


# ============================================================
# ANNUAL EXCEPTION BREAKDOWN
# ============================================================

annual = baseline.copy()

annual["Year"] = (
    annual["Date"].dt.year
)

annual_summary = (
    annual
    .groupby("Year")
    .agg(
        Observations=(
            "Date",
            "size",
        ),
        Normal_Exceptions=(
            "Exception_N",
            "sum",
        ),
        StudentT_Exceptions=(
            "Exception_T",
            "sum",
        ),
    )
    .reset_index()
)

annual_summary[
    "Normal_Exception_Rate"
] = (
    annual_summary[
        "Normal_Exceptions"
    ]
    /
    annual_summary[
        "Observations"
    ]
)

annual_summary[
    "StudentT_Exception_Rate"
] = (
    annual_summary[
        "StudentT_Exceptions"
    ]
    /
    annual_summary[
        "Observations"
    ]
)

annual_summary.to_csv(
    OUT_DIR
    / "garch_exceptions_by_year.csv",
    index=False,
)

print()
print(
    "ANNUAL STUDENT-T EXCEPTIONS"
)
print("--------------------------------")

print(
    annual_summary[
        [
            "Year",
            "Observations",
            "StudentT_Exceptions",
            "StudentT_Exception_Rate",
        ]
    ].to_string(
        index=False,
        formatters={
            "StudentT_Exception_Rate":
                "{:.2%}".format
        },
    )
)


# ============================================================
# STATISTICAL BACKTESTING
# ============================================================

statistical_rows = []

for (
    name,
    var_col,
) in [
    (
        "Normal GARCH(1,1)",
        "VaR_N",
    ),
    (
        "Student-t GARCH(1,1)",
        "VaR_T",
    ),
]:

    result = model_stats(
        baseline,
        var_col,
    )

    statistical_rows.append({
        "Model":
            name,
        "Kupiec_LR":
            result["Kupiec_LR"],
        "Kupiec_p":
            result["Kupiec_p"],
        "Kupiec_Result_5pct":
            (
                "Reject H0"
                if result["Kupiec_p"] < 0.05
                else "Do not reject H0"
            ),
        "Christoffersen_LR":
            result["Christoffersen_LR"],
        "Christoffersen_p":
            result["Christoffersen_p"],
        "Christoffersen_Result_5pct":
            (
                "Reject H0"
                if result[
                    "Christoffersen_p"
                ] < 0.05
                else "Do not reject H0"
            ),
        "n00":
            result["n00"],
        "n01":
            result["n01"],
        "n10":
            result["n10"],
        "n11":
            result["n11"],
    })

statistical_tests = (
    pd.DataFrame(
        statistical_rows
    )
)

statistical_tests.to_csv(
    OUT_DIR
    / "garch_statistical_tests.csv",
    index=False,
)

print()
print("STATISTICAL BACKTESTING")
print("--------------------------------")
print(
    statistical_tests
    .to_string(index=False)
)


# ============================================================
# BACKTESTING CHART
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 5)
)

ax.plot(
    baseline["Date"],
    baseline["Portfolio_PnL"],
    linewidth=0.45,
    label="Portfolio P&L",
)

ax.plot(
    baseline["Date"],
    -baseline["VaR_T"],
    linewidth=1.1,
    label="Student-t GARCH 99% VaR",
)

exceptions_t = baseline[
    baseline["Exception_T"]
]

ax.scatter(
    exceptions_t["Date"],
    exceptions_t["Portfolio_PnL"],
    s=18,
    label="Exceptions",
)

ax.set_title(
    "Student-t GARCH VaR Backtesting"
)

ax.set_ylabel(
    "Dollar P&L / VaR Threshold"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_studentt_backtesting_exceptions.png",
    dpi=180,
)

plt.show()


# ============================================================
# DISTRIBUTION SENSITIVITY
# ============================================================

specification_rows = []

for (
    name,
    var_col,
) in [
    (
        "Normal GARCH(1,1)",
        "VaR_N",
    ),
    (
        "Student-t GARCH(1,1)",
        "VaR_T",
    ),
]:

    result = model_stats(
        baseline,
        var_col,
    )

    specification_rows.append({
        "Configuration":
            name,
        "Average_VaR":
            result["Average_VaR"],
        "Maximum_VaR":
            result["Maximum_VaR"],
        "Exceptions":
            result["Exceptions"],
        "Exception_Rate":
            result["Exception_Rate"],
    })

specification_table = (
    pd.DataFrame(
        specification_rows
    )
)

specification_table.to_csv(
    OUT_DIR
    / "garch_distribution_sensitivity.csv",
    index=False,
)

print()
print("DISTRIBUTION SENSITIVITY")
print("--------------------------------")

print(
    specification_table.to_string(
        index=False,
        formatters={
            "Average_VaR":
                "${:,.2f}".format,
            "Maximum_VaR":
                "${:,.2f}".format,
            "Exception_Rate":
                "{:.2%}".format,
        },
    )
)


# ============================================================
# WINDOW SENSITIVITY
# ============================================================

print()
print(
    "CALCULATING 125-DAY "
    "STUDENT-T GARCH..."
)

garch_t_125 = fit_garch_series(
    window=125,
    distribution="StudentsT",
    refit_every=5,
    label="Student-t 125-day",
)

print()
print(
    "CALCULATING 500-DAY "
    "STUDENT-T GARCH..."
)

garch_t_500 = fit_garch_series(
    window=500,
    distribution="StudentsT",
    refit_every=5,
    label="Student-t 500-day",
)


sensitivity = (
    garch_t_125[
        [
            "Date",
            "Portfolio_PnL",
            "VaR",
        ]
    ]
    .rename(
        columns={
            "VaR":
                "VaR_125"
        }
    )
    .merge(
        garch_t_250[
            [
                "Date",
                "VaR",
            ]
        ].rename(
            columns={
                "VaR":
                    "VaR_250"
            }
        ),
        on="Date",
        how="inner",
    )
    .merge(
        garch_t_500[
            [
                "Date",
                "VaR",
            ]
        ].rename(
            columns={
                "VaR":
                    "VaR_500"
            }
        ),
        on="Date",
        how="inner",
    )
)


sensitivity_rows = []

for (
    label,
    column,
) in [
    (
        "125-day window",
        "VaR_125",
    ),
    (
        "250-day window (baseline)",
        "VaR_250",
    ),
    (
        "500-day window",
        "VaR_500",
    ),
]:

    result = model_stats(
        sensitivity,
        column,
    )

    sensitivity_rows.append({
        "Configuration":
            label,
        "Average_VaR":
            result["Average_VaR"],
        "Maximum_VaR":
            result["Maximum_VaR"],
        "Exceptions":
            result["Exceptions"],
        "Exception_Rate":
            result["Exception_Rate"],
    })


window_sensitivity_table = (
    pd.DataFrame(
        sensitivity_rows
    )
)

baseline_avg_common = (
    window_sensitivity_table.loc[
        window_sensitivity_table[
            "Configuration"
        ]
        ==
        "250-day window (baseline)",
        "Average_VaR",
    ]
    .iloc[0]
)

window_sensitivity_table[
    "Average_VaR_Change_vs_Baseline"
] = (
    window_sensitivity_table[
        "Average_VaR"
    ]
    -
    baseline_avg_common
)

window_sensitivity_table[
    "Average_VaR_Change_Pct"
] = (
    window_sensitivity_table[
        "Average_VaR_Change_vs_Baseline"
    ]
    /
    baseline_avg_common
)

window_sensitivity_table.to_csv(
    OUT_DIR
    / "garch_window_sensitivity.csv",
    index=False,
)

print()
print(
    "ESTIMATION WINDOW SENSITIVITY"
)
print("--------------------------------")
print(
    f"Common observations: "
    f"{len(sensitivity):,}"
)

print(
    window_sensitivity_table.to_string(
        index=False,
        formatters={
            "Average_VaR":
                "${:,.2f}".format,
            "Maximum_VaR":
                "${:,.2f}".format,
            "Exception_Rate":
                "{:.2%}".format,
            "Average_VaR_Change_vs_Baseline":
                "${:,.2f}".format,
            "Average_VaR_Change_Pct":
                "{:.2%}".format,
        },
    )
)


# ============================================================
# WINDOW SENSITIVITY COVID CHART
# ============================================================

covid_sensitivity = sensitivity[
    (
        sensitivity["Date"]
        >= COVID_START
    )
    &
    (
        sensitivity["Date"]
        <= COVID_END
    )
].copy()

fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    covid_sensitivity["Date"],
    covid_sensitivity["VaR_125"],
    label="125-day",
)

ax.plot(
    covid_sensitivity["Date"],
    covid_sensitivity["VaR_250"],
    label="250-day baseline",
)

ax.plot(
    covid_sensitivity["Date"],
    covid_sensitivity["VaR_500"],
    label="500-day",
)

ax.set_title(
    "Student-t GARCH Estimation Window Sensitivity: COVID-19"
)

ax.set_ylabel(
    "99% VaR ($)"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_window_sensitivity_covid.png",
    dpi=180,
)

plt.show()


# ============================================================
# BENCHMARKING AGAINST OTHER MODELS
# ============================================================

benchmark = combined.merge(
    baseline[
        [
            "Date",
            "VaR_T",
            "VaR_N",
        ]
    ],
    on="Date",
    how="inner",
)

benchmark_models = {
    "Historical":
        "Historical_VaR",
    "Parametric":
        "Parametric_VaR",
    "EWMA":
        "EWMA_VaR",
    "GARCH Student-t":
        "VaR_T",
}

benchmark_rows = []

for (
    name,
    column,
) in benchmark_models.items():

    result = model_stats(
        benchmark,
        column,
    )

    benchmark_rows.append({
        "Model":
            name,
        "Average_VaR":
            result["Average_VaR"],
        "Maximum_VaR":
            result["Maximum_VaR"],
        "Exceptions":
            result["Exceptions"],
        "Exception_Rate":
            result["Exception_Rate"],
        "Kupiec_p":
            result["Kupiec_p"],
        "Christoffersen_p":
            result[
                "Christoffersen_p"
            ],
    })

benchmark_table = pd.DataFrame(
    benchmark_rows
)

benchmark_table.to_csv(
    OUT_DIR
    / "garch_model_benchmark_table.csv",
    index=False,
)

print()
print("MODEL BENCHMARKING")
print("--------------------------------")

print(
    benchmark_table.to_string(
        index=False,
        formatters={
            "Average_VaR":
                "${:,.2f}".format,
            "Maximum_VaR":
                "${:,.2f}".format,
            "Exception_Rate":
                "{:.2%}".format,
            "Kupiec_p":
                "{:.6g}".format,
            "Christoffersen_p":
                "{:.6g}".format,
        },
    )
)


# ============================================================
# FULL-PERIOD MODEL COMPARISON
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 5)
)

for (
    name,
    column,
) in benchmark_models.items():

    ax.plot(
        benchmark["Date"],
        -benchmark[column],
        linewidth=1,
        label=name,
    )

ax.set_title(
    "99% VaR Model Comparison: Full Period"
)

ax.set_ylabel(
    "VaR Loss Threshold ($)"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_model_comparison_full_period.png",
    dpi=180,
)

plt.show()


# ============================================================
# COVID STRESS ANALYSIS
# ============================================================

covid = benchmark[
    (
        benchmark["Date"]
        >= COVID_START
    )
    &
    (
        benchmark["Date"]
        <= COVID_END
    )
].copy()

covid_rows = []

for (
    name,
    column,
) in benchmark_models.items():

    result = model_stats(
        covid,
        column,
    )

    peak_index = (
        covid[column].idxmax()
    )

    covid_rows.append({
        "Model":
            name,
        "Average_COVID_VaR":
            result["Average_VaR"],
        "Maximum_COVID_VaR":
            result["Maximum_VaR"],
        "Peak_VaR_Date":
            covid.loc[
                peak_index,
                "Date",
            ],
        "COVID_Exceptions":
            result["Exceptions"],
        "COVID_Exception_Rate":
            result["Exception_Rate"],
    })


covid_comparison = (
    pd.DataFrame(
        covid_rows
    )
)

covid_comparison.to_csv(
    OUT_DIR
    / "garch_covid_model_comparison.csv",
    index=False,
)

print()
print(
    "COVID BASELINE MODEL COMPARISON"
)
print("--------------------------------")

print(
    covid_comparison.to_string(
        index=False,
        formatters={
            "Average_COVID_VaR":
                "${:,.2f}".format,
            "Maximum_COVID_VaR":
                "${:,.2f}".format,
            "COVID_Exception_Rate":
                "{:.2%}".format,
        },
    )
)


# ============================================================
# COVID GARCH P&L VS VAR CHART
# ============================================================

covid[
    "GARCH_Exception"
] = (
    -covid[
        "Portfolio_PnL"
    ]
    >
    covid[
        "VaR_T"
    ]
)

fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    covid["Date"],
    covid["Portfolio_PnL"],
    label="Portfolio P&L",
)

ax.plot(
    covid["Date"],
    -covid["VaR_T"],
    linewidth=1.7,
    label="Student-t GARCH 99% VaR",
)

covid_exceptions = covid[
    covid["GARCH_Exception"]
]

ax.scatter(
    covid_exceptions["Date"],
    covid_exceptions[
        "Portfolio_PnL"
    ],
    s=45,
    label="Exceptions",
    zorder=5,
)

ax.set_title(
    "Student-t GARCH During COVID-19 Stress Period"
)

ax.set_ylabel(
    "Dollar P&L / VaR Threshold"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_covid_stress_pnl_vs_var.png",
    dpi=180,
)

plt.show()


# ============================================================
# COVID MODEL COMPARISON CHART
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 5)
)

for (
    name,
    column,
) in benchmark_models.items():

    ax.plot(
        covid["Date"],
        covid[column],
        linewidth=1.5,
        label=name,
    )

ax.set_title(
    "VaR Model Comparison During COVID-19"
)

ax.set_ylabel(
    "99% VaR ($)"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_covid_model_comparison.png",
    dpi=180,
)

plt.show()


# ============================================================
# RESPONSE AFTER LARGEST COVID LOSSES
# ============================================================

garch_stress = baseline[
    (
        baseline["Date"]
        >= COVID_START
    )
    &
    (
        baseline["Date"]
        <= COVID_END
    )
].copy()

garch_stress[
    "Next_Day_VaR"
] = (
    garch_stress[
        "VaR_T"
    ].shift(-1)
)

garch_stress[
    "Next_Day_VaR_Change"
] = (
    garch_stress[
        "Next_Day_VaR"
    ]
    -
    garch_stress[
        "VaR_T"
    ]
)

garch_stress[
    "Next_Day_VaR_Change_Pct"
] = (
    garch_stress[
        "Next_Day_VaR_Change"
    ]
    /
    garch_stress[
        "VaR_T"
    ]
)

largest_losses = (
    garch_stress
    .nsmallest(
        5,
        "Portfolio_PnL",
    )
    [
        [
            "Date",
            "Portfolio_PnL",
            "VaR_T",
            "Next_Day_VaR",
            "Next_Day_VaR_Change",
            "Next_Day_VaR_Change_Pct",
        ]
    ]
    .sort_values("Date")
)

largest_losses.to_csv(
    OUT_DIR
    / "garch_covid_shock_response.csv",
    index=False,
)

print()
print(
    "GARCH RESPONSE AFTER FIVE "
    "LARGEST COVID LOSS DAYS"
)
print("--------------------------------")

print(
    largest_losses.to_string(
        index=False,
        formatters={
            "Portfolio_PnL":
                "${:,.2f}".format,
            "VaR_T":
                "${:,.2f}".format,
            "Next_Day_VaR":
                "${:,.2f}".format,
            "Next_Day_VaR_Change":
                "${:,.2f}".format,
            "Next_Day_VaR_Change_Pct":
                "{:.2%}".format,
        },
    )
)


# ============================================================
# DAILY REFIT COVID SENSITIVITY
# ============================================================

def daily_refit_covid_student_t():

    rows = []

    target_dates = ret_df.index[
        (
            ret_df.index
            >= COVID_START
        )
        &
        (
            ret_df.index
            <= COVID_END
        )
    ]

    for date in target_dates:

        i = ret_df.index.get_loc(
            date
        )

        if i < BASELINE_WINDOW:
            continue

        window_returns = (
            100
            * ret_df[
                "SPY_Return"
            ].iloc[
                i-BASELINE_WINDOW:i
            ]
        )

        model = arch_model(
            window_returns,
            mean="Constant",
            vol="Garch",
            p=1,
            q=1,
            dist="StudentsT",
        )

        fit = model.fit(
            disp="off",
            show_warning=False,
        )

        forecast = fit.forecast(
            horizon=1,
            reindex=False,
        )

        mu = (
            forecast
            .mean
            .values[-1, 0]
        )

        sigma = np.sqrt(
            forecast
            .variance
            .values[-1, 0]
        )

        nu = fit.params["nu"]

        q = (
            t.ppf(
                1 - CONFIDENCE_LEVEL,
                nu,
            )
            * np.sqrt(
                (nu - 2)
                / nu
            )
        )

        var = (
            -(
                mu
                + sigma * q
            )
            / 100
            * PORTFOLIO_VALUE
        )

        rows.append({
            "Date":
                date,
            "Portfolio_PnL":
                ret_df.loc[
                    date,
                    "Portfolio_PnL",
                ],
            "Daily_Refit_VaR":
                var,
        })

    return pd.DataFrame(
        rows
    )


print()
print(
    "CALCULATING DAILY-REFIT "
    "COVID SENSITIVITY..."
)

daily_refit_covid = (
    daily_refit_covid_student_t()
)


# ============================================================
# CORRECTED MERGE
# ============================================================

refit_comparison = (
    garch_stress[
        [
            "Date",
            "Portfolio_PnL",
            "VaR_T",
        ]
    ]
    .merge(
        daily_refit_covid[
            [
                "Date",
                "Daily_Refit_VaR",
            ]
        ],
        on="Date",
        how="inner",
    )
)


refit_rows = []

for (
    label,
    column,
) in [
    (
        "5-day refit baseline",
        "VaR_T",
    ),
    (
        "Daily refit",
        "Daily_Refit_VaR",
    ),
]:

    result = model_stats(
        refit_comparison,
        column,
    )

    refit_rows.append({
        "Configuration":
            label,
        "Average_COVID_VaR":
            result["Average_VaR"],
        "Maximum_COVID_VaR":
            result["Maximum_VaR"],
        "COVID_Exceptions":
            result["Exceptions"],
        "COVID_Exception_Rate":
            result["Exception_Rate"],
    })


refit_summary = pd.DataFrame(
    refit_rows
)

refit_summary.to_csv(
    OUT_DIR
    / "garch_covid_refit_sensitivity.csv",
    index=False,
)

print()
print(
    "COVID REFIT-FREQUENCY SENSITIVITY"
)
print("--------------------------------")

print(
    refit_summary.to_string(
        index=False,
        formatters={
            "Average_COVID_VaR":
                "${:,.2f}".format,
            "Maximum_COVID_VaR":
                "${:,.2f}".format,
            "COVID_Exception_Rate":
                "{:.2%}".format,
        },
    )
)


# ============================================================
# PARAMETER STABILITY
# ============================================================

parameter_data = (
    baseline.copy()
)

parameter_data[
    "Persistence_N"
] = (
    parameter_data[
        "Alpha_N"
    ]
    +
    parameter_data[
        "Beta_N"
    ]
)

parameter_data[
    "Persistence_T"
] = (
    parameter_data[
        "Alpha_T"
    ]
    +
    parameter_data[
        "Beta_T"
    ]
)


parameter_stability = pd.DataFrame([
    {
        "Metric":
            "Student-t alpha",
        "Mean":
            parameter_data[
                "Alpha_T"
            ].mean(),
        "Median":
            parameter_data[
                "Alpha_T"
            ].median(),
        "Minimum":
            parameter_data[
                "Alpha_T"
            ].min(),
        "Maximum":
            parameter_data[
                "Alpha_T"
            ].max(),
    },
    {
        "Metric":
            "Student-t beta",
        "Mean":
            parameter_data[
                "Beta_T"
            ].mean(),
        "Median":
            parameter_data[
                "Beta_T"
            ].median(),
        "Minimum":
            parameter_data[
                "Beta_T"
            ].min(),
        "Maximum":
            parameter_data[
                "Beta_T"
            ].max(),
    },
    {
        "Metric":
            "Student-t persistence (alpha + beta)",
        "Mean":
            parameter_data[
                "Persistence_T"
            ].mean(),
        "Median":
            parameter_data[
                "Persistence_T"
            ].median(),
        "Minimum":
            parameter_data[
                "Persistence_T"
            ].min(),
        "Maximum":
            parameter_data[
                "Persistence_T"
            ].max(),
    },
    {
        "Metric":
            "Student-t degrees of freedom",
        "Mean":
            parameter_data[
                "Degree_of_Freedom"
            ].mean(),
        "Median":
            parameter_data[
                "Degree_of_Freedom"
            ].median(),
        "Minimum":
            parameter_data[
                "Degree_of_Freedom"
            ].min(),
        "Maximum":
            parameter_data[
                "Degree_of_Freedom"
            ].max(),
    },
])


parameter_stability.to_csv(
    OUT_DIR
    / "garch_parameter_stability.csv",
    index=False,
)

persistence_ge_1 = int(
    (
        parameter_data[
            "Persistence_T"
        ]
        >= 1
    ).sum()
)

persistence_ge_1_rate = (
    persistence_ge_1
    /
    len(parameter_data)
)

print()
print("PARAMETER STABILITY")
print("--------------------------------")

print(
    parameter_stability
    .to_string(index=False)
)

print()

print(
    "Student-t observations with "
    "alpha + beta >= 1:",
    f"{persistence_ge_1:,}",
)

print(
    "Rate with alpha + beta >= 1:",
    f"{persistence_ge_1_rate:.2%}",
)


# ============================================================
# PARAMETER STABILITY CHART
# ============================================================

fig, ax = plt.subplots(
    figsize=(13, 5)
)

ax.plot(
    parameter_data["Date"],
    parameter_data["Alpha_T"],
    label="Alpha",
    linewidth=1,
)

ax.plot(
    parameter_data["Date"],
    parameter_data["Beta_T"],
    label="Beta",
    linewidth=1,
)

ax.plot(
    parameter_data["Date"],
    parameter_data[
        "Persistence_T"
    ],
    label="Alpha + Beta",
    linewidth=1.2,
)

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
    label="Persistence = 1",
)

ax.set_title(
    "Student-t GARCH Parameter Stability"
)

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    FIG_DIR
    / "garch_parameter_stability.png",
    dpi=180,
)

plt.show()


# ============================================================
# FINAL VALIDATION SUMMARY
# ============================================================

student_t_stats = model_stats(
    baseline,
    "VaR_T",
)

covid_garch_stats = model_stats(
    covid,
    "VaR_T",
)

summary = pd.DataFrame(
    [
        [
            "Baseline model",
            "Student-t GARCH(1,1)",
        ],
        [
            "Baseline window",
            "250 trading days",
        ],
        [
            "Baseline refit frequency",
            "Every 5 trading days",
        ],
        [
            "Observations",
            student_t_stats[
                "Observations"
            ],
        ],
        [
            "Average VaR",
            student_t_stats[
                "Average_VaR"
            ],
        ],
        [
            "Maximum VaR",
            student_t_stats[
                "Maximum_VaR"
            ],
        ],
        [
            "Exceptions",
            student_t_stats[
                "Exceptions"
            ],
        ],
        [
            "Exception rate",
            student_t_stats[
                "Exception_Rate"
            ],
        ],
        [
            "Kupiec p-value",
            student_t_stats[
                "Kupiec_p"
            ],
        ],
        [
            "Christoffersen p-value",
            student_t_stats[
                "Christoffersen_p"
            ],
        ],
        [
            "COVID exceptions",
            covid_garch_stats[
                "Exceptions"
            ],
        ],
        [
            "COVID exception rate",
            covid_garch_stats[
                "Exception_Rate"
            ],
        ],
        [
            "Persistence >= 1 count",
            persistence_ge_1,
        ],
        [
            "Persistence >= 1 rate",
            persistence_ge_1_rate,
        ],
    ],
    columns=[
        "Metric",
        "Value",
    ],
)

summary.to_csv(
    OUT_DIR
    / "garch_validation_summary.csv",
    index=False,
)

print()
print(
    "FINAL GARCH VALIDATION SUMMARY"
)
print("--------------------------------")

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# FILES SAVED
# ============================================================

print()
print("FILES SAVED")
print("--------------------------------")

for path in sorted(
    OUT_DIR.glob("*.csv")
):
    print(path)

for path in sorted(
    FIG_DIR.glob("*.png")
):
    print(path)

print()
print(
    "GARCH full validation analysis "
    "completed successfully."
)